# Layer 06 - RAG Search Verification

This notebook verifies the current `03_rag_search_gemma.ipynb` flow with a fixed 100-case golden set.

Scope:
- Search verification compares stored expected Neo4j retrieval results against fresh Neo4j retrieval results.
- Chat verification is an optional groundedness pass that runs only when local Ollama/Gemma is available.

Rules used for this notebook:
- Keep work inside `06_search_layer`
- Use `.venv/bin/python`
- Treat the golden set as a snapshot baseline of the live graph on the generation date

In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

from yc_property_search import Neo4jPropertySearch
from yc_rag_search_gemma import PropertyRAGSearchGemma

print('Python executable:', sys.executable)
print('Python version   :', sys.version.split()[0])

In [ ]:
LAYER_DIR = Path.cwd()
if LAYER_DIR.name != '06_search_layer':
    LAYER_DIR = LAYER_DIR / '06_search_layer'
ARTIFACT_DIR = LAYER_DIR / 'artifacts'
golden_paths = sorted(ARTIFACT_DIR.glob('rag_search_gemma_golden_set_*.json'))
if not golden_paths:
    raise FileNotFoundError(f'No golden set found in {ARTIFACT_DIR}')
golden_path = golden_paths[-1]
payload = json.loads(golden_path.read_text())
golden_cases = payload['cases']
golden_meta = payload['metadata']
print('Golden set:', golden_path.name)
print('Generated :', golden_meta['generated_on'])
print('Case count :', len(golden_cases))
pd.DataFrame(golden_meta['category_counts'].items(), columns=['category', 'count']).sort_values('category')

## 1. Neo4j KG Overview

Use the live graph to confirm that the current database still matches the snapshot used to build the golden set.

In [ ]:
with Neo4jPropertySearch() as neo:
    live_node_counts = neo.node_counts()
    live_schools = neo.graph_query('MATCH (s:FamousSchool) RETURN s.name AS name ORDER BY name')
    live_towns = neo.graph_query('MATCH (p:Property) RETURN p.town AS town, count(*) AS property_count ORDER BY property_count DESC, town ASC')

print('Golden node counts:', golden_meta['node_counts'])
print('Live node counts  :', live_node_counts)
print('Node counts match :', golden_meta['node_counts'] == live_node_counts)

display(pd.DataFrame(live_towns).head(10))
display(pd.DataFrame(live_schools, columns=['name']))

## 2. Search Evaluation

This section bypasses the LLM and compares expected Stage 2 Neo4j retrieval results with fresh Neo4j retrieval results.

Metrics:
- `exact_top1_match`: first expected address equals first actual address
- `top3_recall`: overlap between expected top 3 and actual top 3
- `top5_recall`: overlap between expected top 5 and actual top 5
- `pass`: strict regression pass for the stored snapshot

In [ ]:
def run_case(neo: Neo4jPropertySearch, case: dict) -> pd.DataFrame:
    params = case['params']
    if case['search_mode'] == 'near_famous_school':
        school_name = params['filters']['school_name']
        cypher = '''\
MATCH (p:Property)-[r:NEAR_FAMOUS_SCHOOL]->(s:FamousSchool {name: $school_name})
RETURN p.address_key AS address_key,
       p.town AS town,
       p.flat_type AS flat_type,
       p.resale_price AS resale_price,
       p.floor_area_sqm AS floor_area_sqm,
       p.lease_remaining_years AS lease_remaining_years,
       p.dist_to_mrt_m AS dist_to_mrt_m,
       p.nearest_famous_school_name AS nearest_famous_school_name,
       r.distance_km AS dist_to_nearest_famous_school_km
ORDER BY r.distance_km ASC, p.resale_price ASC
LIMIT $top_k
'''
        rows = neo.graph_query(cypher, school_name=school_name, top_k=case['top_k'])
        return pd.DataFrame(rows)
    return neo.search(
        weights=params['weights'],
        filters=params['filters'],
        top_k=case['top_k'],
    )

def overlap_ratio(expected: list[str], actual: list[str]) -> float:
    if not expected:
        return 1.0 if not actual else 0.0
    return len(set(expected) & set(actual)) / len(expected)

def compare_case(neo: Neo4jPropertySearch, case: dict) -> dict:
    actual = run_case(neo, case)
    expected_addresses = case['expected']['top_addresses']
    actual_addresses = actual['address_key'].tolist() if 'address_key' in actual.columns else []
    exact_top1_match = (expected_addresses[:1] == actual_addresses[:1])
    top3_recall = overlap_ratio(expected_addresses[:3], actual_addresses[:3])
    top5_recall = overlap_ratio(expected_addresses[:5], actual_addresses[:5])
    expected_empty = case['query_type'] == 'negative'
    pass_flag = ((expected_empty and actual.empty) or (not expected_empty and exact_top1_match and top3_recall == 1.0 and top5_recall == 1.0))
    return {
        'id': case['id'],
        'category': case['category'],
        'question': case['question'],
        'search_mode': case['search_mode'],
        'expected_empty': expected_empty,
        'actual_count': len(actual),
        'expected_top1': expected_addresses[0] if expected_addresses else None,
        'actual_top1': actual_addresses[0] if actual_addresses else None,
        'exact_top1_match': exact_top1_match,
        'top3_recall': top3_recall,
        'top5_recall': top5_recall,
        'pass': pass_flag,
    }


In [ ]:
with Neo4jPropertySearch() as neo:
    search_eval = pd.DataFrame(compare_case(neo, case) for case in golden_cases)

summary = pd.DataFrame([
    {
        'total_cases': len(search_eval),
        'strict_pass_rate': round(search_eval['pass'].mean(), 4),
        'top1_accuracy': round(search_eval['exact_top1_match'].mean(), 4),
        'avg_top3_recall': round(search_eval['top3_recall'].mean(), 4),
        'avg_top5_recall': round(search_eval['top5_recall'].mean(), 4),
        'negative_cases': int(search_eval['expected_empty'].sum()),
        'positive_cases': int((~search_eval['expected_empty']).sum()),
    }
])
display(summary)
display(search_eval.groupby('category')[['pass', 'exact_top1_match', 'top3_recall', 'top5_recall']].mean().reset_index())

In [ ]:
search_failures = search_eval.loc[~search_eval['pass']].copy()
display(search_failures.head(20))
print('Failure count:', len(search_failures))

## 3. Optional Chat Evaluation

This section verifies the chat layer only. It reuses the evaluated Neo4j search results from Section 2 and calls the Gemma Stage 3 answer generator.

Groundedness checks are intentionally lightweight because answer wording is generative:
- non-empty answer
- empty-result cases should use the standard no-match response
- positive-result answers should mention at least one stored answer anchor such as the town, school, or top address

In [ ]:
def ollama_available() -> bool:
    try:
        import requests
        response = requests.get('http://localhost:11434/api/tags', timeout=3)
        return response.ok
    except Exception:
        return False

OLLAMA_READY = ollama_available()
print('Ollama ready:', OLLAMA_READY)

In [ ]:
def evaluate_chat_case(neo: Neo4jPropertySearch, rag: PropertyRAGSearchGemma, case: dict) -> dict:
    actual_results = run_case(neo, case)
    answer = rag._stage3_generate_answer(case['question'], actual_results) or ''
    anchors = [anchor for anchor in case['expected']['answer_anchors'] if anchor]
    anchor_hit = any(anchor.lower() in answer.lower() for anchor in anchors)
    empty_case = case['query_type'] == 'negative'
    empty_response_ok = ('no matching properties were found' in answer.lower()) if empty_case else None
    return {
        'id': case['id'],
        'category': case['category'],
        'results_empty': bool(actual_results.empty),
        'answer_non_empty': bool(answer.strip()),
        'anchor_hit': anchor_hit,
        'empty_response_ok': empty_response_ok,
        'pass': (bool(answer.strip()) and ((empty_case and bool(empty_response_ok)) or ((not empty_case) and anchor_hit))),
        'answer': answer,
    }

if OLLAMA_READY:
    neo = Neo4jPropertySearch()
    rag = PropertyRAGSearchGemma(ollama_base_url='http://localhost:11434', top_k_results=5)
    try:
        chat_eval = pd.DataFrame(evaluate_chat_case(neo, rag, case) for case in golden_cases)
    finally:
        neo.close()
        rag.close()
    display(pd.DataFrame([{
        'total_cases': len(chat_eval),
        'chat_pass_rate': round(chat_eval['pass'].mean(), 4),
        'anchor_hit_rate': round(chat_eval['anchor_hit'].mean(), 4),
    }]))
    display(chat_eval.loc[~chat_eval['pass'], ['id', 'category', 'results_empty', 'anchor_hit', 'empty_response_ok', 'answer']].head(20))
else:
    print('Ollama/Gemma is not available in this environment, so chat evaluation is skipped.')